# Notebook 4: DistilBERT fine-tuning

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook fine-tunes a pretrained Transformer and compares it with the TF-IDF baseline and TextCNN.

## Experiment design

- Fine-tune `distilbert-base-uncased` separately on SMS and Enron.
- Use the existing train, validation, and test splits.
- Truncate inputs to 256 tokens and use dynamic padding per batch.
- Apply balanced class weights and select the best checkpoint by validation F1.
- Evaluate both models on both test domains without test-set tuning.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers>=4.57,<5', 'accelerate>=1.10,<2',
        ],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import transformers
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import load_prepared_splits, summarize_splits
from src.distilbert import run_distilbert_experiments
from src.modeling import run_transfer_experiments

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 42

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'GPU devices: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')

if IS_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running this notebook.')

## Load the prepared data

The preprocessing and splits are identical to the previous notebooks.

In [ ]:
splits, cleaning_audit = load_prepared_splits(random_state=RANDOM_STATE)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Fixed fine-tuning configuration

The configuration is deliberately small enough for a Kaggle GPU session and is kept unchanged between domains. Batch sizes are specified per GPU.

In [ ]:
DISTILBERT_CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'random_state': RANDOM_STATE,
    'max_length': 256,
    'learning_rate': 2e-5,
    'epochs': 2,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'require_gpu': True,
}

pd.Series(DISTILBERT_CONFIG, name='value').to_frame()

## Fine-tune on each domain

This is the only long-running cell. The SMS model is evaluated and removed from GPU memory before the Enron model is created. A warning that the classification head is newly initialized is expected.

In [ ]:
training_histories, distilbert_results, distilbert_details = (
    run_distilbert_experiments(splits, **DISTILBERT_CONFIG)
)
print('Finished fine-tuning the SMS and Enron DistilBERT models.')

## Learning curves

Training and validation loss are checked for instability or immediate overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, domain in zip(axes, ['sms', 'enron']):
    history = training_histories[domain]
    train_rows = history.dropna(subset=['loss'])
    validation_rows = history.dropna(subset=['eval_loss'])
    axis.plot(train_rows['epoch'], train_rows['loss'], marker='o', label='train')
    axis.plot(
        validation_rows['epoch'],
        validation_rows['eval_loss'],
        marker='o',
        label='validation',
    )
    axis.set_title(f'{domain.upper()} fine-tuning loss')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Cross-entropy')
    axis.legend()

plt.tight_layout()
plt.show()

## DistilBERT results

F1 remains the primary metric, with ROC-AUC and the error counts providing additional context.

In [ ]:
metric_columns = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
display_results = distilbert_results.copy()
display_results[metric_columns] = display_results[metric_columns].round(3)
display_results['training_seconds'] = display_results['training_seconds'].round(1)
display_results.loc[:, [
    'train_domain', 'test_domain', 'setting', 'test_rows',
    *metric_columns, 'epochs_trained', 'training_seconds', 'parameters',
    'gpu_count', 'effective_train_batch_size',
]]

## Compare all three approaches

The Logistic Regression baseline is refitted here. The TextCNN values are the rounded results from the [executed Notebook 3](https://www.kaggle.com/code/kaloyanbozukov/notebook-3-textcnn-cross-domain-experiment), so that model does not need to be trained again.

In [ ]:
baseline_models, baseline_results = run_transfer_experiments(
    splits, random_state=RANDOM_STATE
)
baseline_results = baseline_results.assign(model='TF-IDF + Logistic Regression')

textcnn_reference = pd.DataFrame([
    {'train_domain': 'sms', 'test_domain': 'sms', 'accuracy': 0.983, 'precision': 0.966, 'recall': 0.896, 'f1': 0.930, 'roc_auc': 0.984},
    {'train_domain': 'sms', 'test_domain': 'enron', 'accuracy': 0.528, 'precision': 0.525, 'recall': 0.606, 'f1': 0.563, 'roc_auc': 0.540},
    {'train_domain': 'enron', 'test_domain': 'sms', 'accuracy': 0.351, 'precision': 0.155, 'recall': 0.948, 'f1': 0.266, 'roc_auc': 0.791},
    {'train_domain': 'enron', 'test_domain': 'enron', 'accuracy': 0.992, 'precision': 0.991, 'recall': 0.994, 'f1': 0.992, 'roc_auc': 1.000},
]).assign(model='TextCNN')

comparison = pd.concat(
    [
        baseline_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
        textcnn_reference.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
        distilbert_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
    ],
    ignore_index=True,
)
comparison['experiment'] = (
    comparison['train_domain'].str.upper()
    + ' -> '
    + comparison['test_domain'].str.upper()
)
experiment_order = [
    'SMS -> SMS', 'SMS -> ENRON', 'ENRON -> SMS', 'ENRON -> ENRON'
]
comparison.pivot(index='experiment', columns='model', values='f1').reindex(
    experiment_order
).round(3)

In [ ]:
plt.figure(figsize=(11, 5))
axis = sns.barplot(
    data=comparison,
    x='experiment',
    y='f1',
    hue='model',
    order=experiment_order,
)
axis.set_title('F1 comparison across models and domains')
axis.set_xlabel('Train -> test domain')
axis.set_ylabel('F1 score')
axis.set_ylim(0, 1)
axis.legend(title='Model', loc='lower right')
for container in axis.containers:
    axis.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

## DistilBERT confusion matrices

The four matrices show how the error balance changes outside the training domain.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, row in zip(axes.ravel(), distilbert_results.itertuples(index=False)):
    details = distilbert_details[(row.train_domain, row.test_domain)]
    ConfusionMatrixDisplay.from_predictions(
        details['label'],
        details['prediction'],
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        f'Train {row.train_domain.upper()} -> Test {row.test_domain.upper()}'
    )

plt.tight_layout()
plt.show()

## SMS -> Enron error analysis

High-confidence errors reveal which email examples remain difficult after pretrained language-model fine-tuning.

In [ ]:
sms_to_enron = distilbert_details[('sms', 'enron')].copy()
cross_domain_errors = sms_to_enron.loc[~sms_to_enron['correct']].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = (
    cross_domain_errors['spam_probability'] - 0.5
).abs()

display(cross_domain_errors['error_type'].value_counts().rename('errors').to_frame())
cross_domain_errors.sort_values('confidence', ascending=False).loc[
    :, ['error_type', 'spam_probability', 'text']
].head(10)

## Final comparison

The main result is whether pretraining improves SMS-to-email transfer and reduces the gap from in-domain SMS performance.

In [ ]:
def f1_for(frame, train_domain, test_domain):
    return frame.query(
        'train_domain == @train_domain and test_domain == @test_domain'
    )['f1'].iloc[0]

baseline_cross_f1 = f1_for(baseline_results, 'sms', 'enron')
textcnn_cross_f1 = f1_for(textcnn_reference, 'sms', 'enron')
distilbert_in_domain_f1 = f1_for(distilbert_results, 'sms', 'sms')
distilbert_cross_f1 = f1_for(distilbert_results, 'sms', 'enron')
distilbert_transfer_gap = distilbert_in_domain_f1 - distilbert_cross_f1

print(f'TF-IDF SMS -> Enron F1:       {baseline_cross_f1:.3f}')
print(f'TextCNN SMS -> Enron F1:       {textcnn_cross_f1:.3f}')
print(f'DistilBERT SMS -> SMS F1:      {distilbert_in_domain_f1:.3f}')
print(f'DistilBERT SMS -> Enron F1:    {distilbert_cross_f1:.3f}')
print(f'DistilBERT transfer gap:       {distilbert_transfer_gap:.3f}')
print(
    'Change vs ML cross-domain:      '
    f'{distilbert_cross_f1 - baseline_cross_f1:+.3f}'
)
print(
    'Change vs TextCNN cross-domain: '
    f'{distilbert_cross_f1 - textcnn_cross_f1:+.3f}'
)

cross_domain_scores = {
    'TF-IDF + Logistic Regression': baseline_cross_f1,
    'TextCNN': textcnn_cross_f1,
    'DistilBERT': distilbert_cross_f1,
}
best_model = max(cross_domain_scores, key=cross_domain_scores.get)
print(f'Best SMS-to-Enron model by F1: {best_model}.')

## Scope and limitations

- Emails longer than 256 tokens are truncated.
- One fixed configuration is used; no broad hyperparameter search is performed.
- Conclusions apply to these two datasets and their specific spam distributions.
- Test sets are used only for the final four evaluations.